# Estudo do modelo: RandomForest

- **Família:** Clássico (tabular)
- **Tipo de entrada (registry):** `tabular`
- **Por quê:** Ensemble de árvores sobre features agregadas; rápido e robusto.

## Objetivos

1. Mostrar o **contrato de entrada** que o benchmark prepara para esta
   arquitetura.
2. Instanciar o modelo pelo **mesmo caminho** do benchmark (`summary()`).
3. **Treinar + avaliar** em dados sintéticos (accuracy/AUC/EER) — ligado
   por padrão (defina `XFAKE_RUN_EVAL=0` para pular).

Para treino com o seu próprio dataset, use
`notebooks/pipeline/02_training_model.ipynb`.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

_REPO_URL = "https://github.com/thierrybraga/XFakeSong.git"


def _project_root() -> Path:
    p = Path.cwd()
    while not (p / "app").exists() and p.parent != p:
        p = p.parent
    return p


# Colab/Kaggle limpos: clona o repositório se o projeto não estiver no disco.
if not (_project_root() / "app").exists():
    if not Path("XFakeSong").exists():
        print("Clonando XFakeSong...")
        subprocess.run(["git", "clone", "--depth", "1", _REPO_URL], check=True)
    os.chdir("XFakeSong")

_DEPS = {"librosa": "librosa>=0.10", "soundfile": "soundfile>=0.12",
         "pywt": "PyWavelets>=1.4"}
_missing = [s for m, s in _DEPS.items() if importlib.util.find_spec(m) is None]
if _missing:
    print("Instalando dependências de áudio:", *_missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing],
                   check=True)

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT, "| Colab:", "google.colab" in sys.modules)

## Reprodutibilidade (semente numpy + TensorFlow)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(0)
except Exception:  # pragma: no cover
    np.random.seed(0)

In [ ]:
ARCHITECTURE = 'RandomForest'

from benchmarks.data import BenchmarkData

data = BenchmarkData.synthetic(n=48, shape=(64, 40), seed=7)
prepared = data.prepare_for_architecture(ARCHITECTURE)
print("Arquitetura     :", ARCHITECTURE)
print("Shape original  :", data.X.shape)
print("Shape preparado :", prepared.X.shape)
print("Metadados       :", prepared.metadata)

## Inspeção do modelo

In [ ]:
ARCHITECTURE = 'RandomForest'

# Modelos clássicos (sklearn) são treinados/avaliados pelo harness de
# benchmark (caminho clássico), não pelo factory Keras. Rodamos um
# benchmark rápido em dados sintéticos só para validar o fluxo.
from benchmarks import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig.quick(
    architectures=[ARCHITECTURE],
    synthetic_n=120,
    snr_levels_db=[20],
    output_dir=str(ROOT / "results" / "notebook_smoke" / ARCHITECTURE.lower()),
)
results = run_benchmark(cfg)
print("clean:", results["architectures"][ARCHITECTURE]["clean"])

## Treino + avaliação em dados sintéticos

In [ ]:
import os

# Treina + avalia este modelo em dados sintéticos (rápido) pelo MESMO harness do
# benchmark do TCC. LIGADO por padrão para o estudo ser funcional de ponta a
# ponta; defina XFAKE_RUN_EVAL=0 para pular (usado no CI de build).
# Em Colab, selecione um runtime com GPU (Ambiente de execução → Alterar o tipo
# de hardware → GPU) para acelerar.
RUN_EVAL = os.environ.get("XFAKE_RUN_EVAL", "1") == "1"

if RUN_EVAL:
    from benchmarks import BenchmarkConfig, run_benchmark

    cfg = BenchmarkConfig.quick(
        architectures=[ARCHITECTURE],
        synthetic_n=160,
        snr_levels_db=[20],
        output_dir=str(ROOT / "results" / "notebook_eval" / ARCHITECTURE.lower()),
    )
    r = run_benchmark(cfg)["architectures"][ARCHITECTURE]
    if r.get("status") == "ok":
        c = r["clean"]
        print(f"accuracy = {c['accuracy'] * 100:5.1f}%")
        print(f"AUC-ROC  = {c.get('auc_roc', float('nan')):.3f}")
        print(f"EER      = {c.get('eer', float('nan')):.3f}")
        print(f"convergiu: {r.get('converged')}")
    else:
        print("ERRO:", r.get("error"))
else:
    print("Treino desligado (XFAKE_RUN_EVAL=0).")

## Como ler este notebook

- `docs/08_ARQUITETURAS.md` — descrição de todas as arquiteturas.
- `docs/10_TREINAMENTO.md` — hiperparâmetros e estratégia de treino.
- `docs/15_BENCHMARK.md` — execução completa, métricas e gráficos.

No relatório, compare sempre: acurácia, EER, AUC-ROC, min-tDCF,
latência, tamanho do modelo, matriz de confusão e distribuição de scores.

## Hiperparâmetros e análise esperada

- **Treino rápido:** use `epochs=2` apenas para validar fluxo.
- **Treino para TCC:** use o preset do benchmark completo em
  `notebooks/pipeline/01_benchmark_tcc_full_pipeline.ipynb`.
- **Entrada preparada:** confira a célula de `BenchmarkData` acima;
  ela mostra o shape real usado no harness.
- **Arquivos esperados no relatório:** `architectures/random_forest/metrics.json`,
  `confusion_matrix.png`, `roc.png`, `score_distribution.png` e,
  quando houver histórico, `convergence.png`.

In [ ]:
import json

metrics_path = ROOT / "results" / 'tcc_full_20k' / "architectures" / 'random_forest' / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics.get("clean", metrics), indent=2, ensure_ascii=False)[:2000])
else:
    print("Sem métricas salvas ainda:", metrics_path)
    print("Rode o benchmark completo ou ajuste o caminho de results.")